# Zusatzanalyse – Kaskade und Fehlerkomplementarität bei 25 % Labels

Dieses Notebook ist eine **nachgelagerte Analyse des FINAL FREEZE**. Es verändert weder
SSL-Training noch Splits noch Downstream-Hyperparameter.

Untersucht werden für jeden Downstream-Klassifikator:

- Logistic Regression
- XGBoost
- MLP

jeweils die Repräsentationen:

- BASE
- DAPT
- CONTRASTIVE

## Kaskadenlogik

Stage 1 ist immer `BASE + gleicher Downstream-Klassifikator` am bereits definierten
operativen Zielpunkt von 0,5 % FPR auf dem Kalibrierungssplit.

Unter den von Stage 1 **nicht** als Phishing erkannten Webseiten werden anschließend
drei Rankingstrategien verglichen:

1. `BASE_SELF`: BASE-Scores selbst (Kontrollbedingung)
2. `DAPT`: Scores der DAPT-Repräsentation
3. `CONTRASTIVE`: Scores der Contrastive-Repräsentation

Bei Reviewbudgets von 5 %, 10 % und 20 % der Stage-1-negativen Fälle wird gemessen,
wie viele zusätzliche Stage-1-False-Negatives in das Review gelangen.

Die zweite Stufe ist damit **Triage/Review und keine automatische Blockentscheidung**.
Die FPR des automatischen Stage-1-Detektors wird dadurch nicht künstlich erhöht.

Zusätzlich werden paarweise Fehlerüberlappung, Rescue-/Regression-Raten und
qualitative Fehlerbeispiele ausgewertet.

In [ ]:
# ============================================================
# 00 – Imports und Konfiguration
# ============================================================
import os, gc, json, math, pickle, shutil, zipfile, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    average_precision_score, roc_auc_score, confusion_matrix,
    precision_score, recall_score, f1_score
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/phreshphish_cascade_error_analysis_25pct")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 52, 62, 72, 82]
REPRESENTATIONS = ["BASE", "DAPT", "CONTRASTIVE"]
CLASSIFIERS = ["LOGREG", "XGBOOST", "MLP"]
LABEL_BUDGET = 0.25
TARGET_FPR = 0.005
CALIBRATION_FRACTION = 0.30
CALIBRATION_SPLIT_SEED = 20260808
REVIEW_FRACTIONS = [0.05, 0.10, 0.20]

print({
    "analysis": "cascade_error_25pct",
    "seeds": SEEDS,
    "classifiers": CLASSIFIERS,
    "review_fractions": REVIEW_FRACTIONS,
})

In [ ]:
# ============================================================
# 01 – FINAL-FREEZE Output und ENDGAME finden
# ============================================================

def all_named(name):
    return list(INPUT_ROOT.rglob(name))

def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def looks_like_freeze_root(p):
    if not p.is_dir():
        return False
    return (
        (p / "embeddings").exists()
        and (p / "best_classifier_params.json").exists()
    )

freeze_candidates = [p for p in INPUT_ROOT.rglob("*") if looks_like_freeze_root(p)]

if not freeze_candidates:
    # ZIP-Fallback
    zips = list(INPUT_ROOT.rglob("*.zip"))
    extracted = Path("/kaggle/working/_freeze_extract")
    if extracted.exists():
        shutil.rmtree(extracted)
    for z in zips:
        try:
            with zipfile.ZipFile(z, "r") as zf:
                names = zf.namelist()
                if any("embeddings/" in n for n in names) and any(
                    "best_classifier_params.json" in n for n in names
                ):
                    extracted.mkdir(parents=True, exist_ok=True)
                    zf.extractall(extracted)
                    break
        except Exception:
            continue
    freeze_candidates = [p for p in extracted.rglob("*") if looks_like_freeze_root(p)]
    if looks_like_freeze_root(extracted):
        freeze_candidates.append(extracted)

if not freeze_candidates:
    raise FileNotFoundError(
        "FINAL-FREEZE Output nicht gefunden. Bitte den vollständigen "
        "phreshphish_final_freeze_output bzw. die Ergebnis-ZIP als Kaggle Input anhängen."
    )

FREEZE_ROOT = sorted(freeze_candidates, key=lambda p: len(str(p)))[0]

split_paths = all_named("split_roles_and_holdout_cache_v2_ram_safe.pkl")
dapt_paths = all_named("dapt40k_bundle.pkl")
if not split_paths or not dapt_paths:
    raise FileNotFoundError("ENDGAME Split-Cache bzw. dapt40k_bundle.pkl fehlt.")

SPLIT_PATH = split_paths[0]
DAPT_PATH = dapt_paths[0]

print({
    "freeze_root": str(FREEZE_ROOT),
    "split_cache": str(SPLIT_PATH),
    "dapt_bundle": str(DAPT_PATH),
})

In [ ]:
# ============================================================
# 02 – Datenrollen und Freeze-Szenarien rekonstruieren
# ============================================================

split_payload = load_pickle(SPLIT_PATH)
dapt_payload = load_pickle(DAPT_PATH)

def first_existing(obj, keys):
    if not isinstance(obj, dict):
        return None
    for k in keys:
        if k in obj:
            return obj[k]
    return None

train_df = first_existing(
    split_payload, ["train_df","train","supervised_train","downstream_train"]
)
val_df = first_existing(
    split_payload, ["val_df","validation_df","validation","val"]
)
holdout_df = first_existing(
    split_payload, ["final_holdout_clean","final_holdout","holdout_df","holdout"]
)
pretrain_df = first_existing(
    dapt_payload, ["pretrain_large_df","frame","pretrain_df","pretrain"]
)
dapt_holdout = first_existing(dapt_payload, ["final_holdout_clean"])

for name, obj in [
    ("train",train_df),("validation",val_df),("holdout",holdout_df),("pretrain",pretrain_df)
]:
    if not isinstance(obj, pd.DataFrame):
        raise TypeError(f"{name} ist kein DataFrame.")

train_df = train_df.reset_index(drop=True).copy()
val_df = val_df.reset_index(drop=True).copy()
holdout_df = holdout_df.reset_index(drop=True).copy()
pretrain_df = pretrain_df.reset_index(drop=True).copy()

# 40k-aware Holdout-Flags übernehmen.
if isinstance(dapt_holdout, pd.DataFrame):
    dapt_holdout = dapt_holdout.reset_index(drop=True)
    lookup = dapt_holdout.set_index(dapt_holdout["sha256"].astype(str))
    hs = holdout_df["sha256"].astype(str)
    for c in [
        "near_duplicate_to_development",
        "min_simhash_distance_to_development",
        "template_seen_in_development",
    ]:
        if c in lookup.columns:
            holdout_df[c] = hs.map(lookup[c]).to_numpy()

cal_idx, iid_idx = train_test_split(
    np.arange(len(val_df)),
    test_size=1.0-CALIBRATION_FRACTION,
    random_state=CALIBRATION_SPLIT_SEED,
    stratify=val_df["label"].to_numpy(),
)
calibration_df = val_df.iloc[np.sort(cal_idx)].reset_index(drop=True)
iid_df = val_df.iloc[np.sort(iid_idx)].reset_index(drop=True)

development_domains = set()
development_templates = set()
for frame in [pretrain_df, train_df, val_df]:
    development_domains.update(frame["domain"].fillna("").astype(str))
    development_templates.update(frame["template_hash"].fillna("").astype(str))
development_domains.discard("")
development_templates.discard("")

h_domain = holdout_df["domain"].fillna("").astype(str)
h_template = holdout_df["template_hash"].fillna("").astype(str)
domain_seen = h_domain.isin(development_domains).to_numpy()
template_seen = h_template.isin(development_templates).to_numpy()
near_dup = holdout_df["near_duplicate_to_development"].fillna(False).astype(bool).to_numpy()

def balanced_indices(frame, mask, seed):
    sub = frame.loc[np.asarray(mask)].copy()
    n = min(int((sub.label==0).sum()), int((sub.label==1).sum()))
    if n <= 0:
        raise RuntimeError("Szenario enthält nicht beide Klassen.")
    a = sub[sub.label.eq(0)].sample(n=n, random_state=seed)
    b = sub[sub.label.eq(1)].sample(n=n, random_state=seed+1)
    return np.sort(pd.concat([a,b]).index.to_numpy(dtype=np.int32))

SCENARIOS = {
    "IID": ("iid", np.arange(len(iid_df), dtype=np.int32)),
    "TEMPORAL": ("holdout", np.arange(len(holdout_df), dtype=np.int32)),
    "DOMAIN_OOD": (
        "holdout",
        balanced_indices(
            holdout_df,
            (~domain_seen) & h_domain.ne("").to_numpy(),
            108,
        ),
    ),
    "TEMPLATE_OOD": (
        "holdout",
        balanced_indices(
            holdout_df,
            (~template_seen) & (~near_dup) & h_template.ne("").to_numpy(),
            109,
        ),
    ),
}
SCENARIOS["DOMAIN_TEMPLATE_OOD"] = (
    "holdout",
    balanced_indices(
        holdout_df,
        (~domain_seen)
        & h_domain.ne("").to_numpy()
        & (~template_seen)
        & (~near_dup)
        & h_template.ne("").to_numpy(),
        110,
    ),
)

expected = {
    "IID": 3051,
    "TEMPORAL": 8000,
    "DOMAIN_OOD": 3858,
    "TEMPLATE_OOD": 6832,
    "DOMAIN_TEMPLATE_OOD": 3708,
}
actual = {k: len(v[1]) for k,v in SCENARIOS.items()}
print({"scenario_rows": actual})
if actual != expected:
    raise RuntimeError(f"Szenariogrößen weichen vom FINAL FREEZE ab: {actual} != {expected}")

In [ ]:
# ============================================================
# 03 – Embeddings, Parameter und 25%-Labelsubset
# ============================================================

with open(FREEZE_ROOT / "best_classifier_params.json", "r", encoding="utf-8") as f:
    BEST_PARAMS = json.load(f)
if "hidden_layer_sizes" in BEST_PARAMS["MLP"]:
    BEST_PARAMS["MLP"]["hidden_layer_sizes"] = tuple(
        BEST_PARAMS["MLP"]["hidden_layer_sizes"]
    )

EMBED_ROOT = FREEZE_ROOT / "embeddings"

def emb_path(rep, seed, split):
    seed_key = "shared" if rep == "BASE" else f"seed_{seed}"
    return EMBED_ROOT / rep.lower() / seed_key / f"{split}.npy"

def load_emb(rep, seed, split):
    s = SEEDS[0] if rep == "BASE" else seed
    p = emb_path(rep, s, split)
    if not p.exists():
        raise FileNotFoundError(p)
    return np.asarray(np.load(p, mmap_mode="r"), dtype=np.float32)

def build_classifier(kind, params, seed):
    if kind == "LOGREG":
        return Pipeline([
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(
                C=params["C"], max_iter=2500, solver="lbfgs", random_state=seed
            )),
        ])
    if kind == "XGBOOST":
        return XGBClassifier(
            **params,
            subsample=0.9,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=2,
            random_state=seed,
        )
    if kind == "MLP":
        return Pipeline([
            ("scale", StandardScaler()),
            ("clf", MLPClassifier(
                **params,
                activation="relu",
                solver="adam",
                batch_size=128,
                learning_rate_init=1e-3,
                max_iter=200,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=12,
                random_state=seed,
            )),
        ])
    raise KeyError(kind)

y_train = train_df["label"].to_numpy(dtype=int)

def budget_25_indices(y, seed):
    rng = np.random.default_rng(seed)
    parts = []
    for c in [0,1]:
        idx = np.flatnonzero(y==c).copy()
        rng.shuffle(idx)
        n = int(round(len(idx) * LABEL_BUDGET))
        parts.append(idx[:n])
    return np.sort(np.concatenate(parts)).astype(np.int32)

BUDGET_IDX = {seed: budget_25_indices(y_train, seed) for seed in SEEDS}
for seed, idx in BUDGET_IDX.items():
    assert len(idx) == 1000
    assert int((y_train[idx]==0).sum()) == 500
    assert int((y_train[idx]==1).sum()) == 500

print({"best_params": BEST_PARAMS, "label_budget_n": 1000})

In [ ]:
# ============================================================
# 04 – Schwellenwert und Vorhersagen
# ============================================================

def threshold_for_target_fpr(y_true, score, target_fpr):
    y = np.asarray(y_true)
    s = np.asarray(score)
    neg = np.sort(s[y==0])[::-1]
    allowed = int(math.floor(target_fpr * len(neg) + 1e-12))
    if allowed <= 0:
        return float(np.nextafter(neg[0], np.inf))
    if allowed >= len(neg):
        return float(-np.inf)
    return float(np.nextafter(neg[allowed], np.inf))

PRED_DIR = OUTPUT_ROOT / "scores"
PRED_DIR.mkdir(exist_ok=True)

prediction_index_rows = []

for seed in SEEDS:
    idx = BUDGET_IDX[seed]
    yb = y_train[idx]
    ycal = calibration_df["label"].to_numpy(dtype=int)

    for kind in CLASSIFIERS:
        for rep in REPRESENTATIONS:
            key = f"{rep.lower()}_{kind.lower()}_seed{seed}"
            out_file = PRED_DIR / f"{key}.npz"

            if out_file.exists():
                z = np.load(out_file)
                prediction_index_rows.append({
                    "seed":seed,"classifier":kind,"representation":rep,
                    "threshold":float(z["threshold"])
                })
                continue

            Xtr = load_emb(rep, seed, "train")[idx]
            Xcal = load_emb(rep, seed, "calibration")

            model = build_classifier(kind, BEST_PARAMS[kind], seed)
            model.fit(Xtr, yb)

            cal_score = model.predict_proba(Xcal)[:,1]
            threshold = threshold_for_target_fpr(ycal, cal_score, TARGET_FPR)

            iid_score = model.predict_proba(load_emb(rep, seed, "iid"))[:,1]
            holdout_score = model.predict_proba(load_emb(rep, seed, "holdout"))[:,1]

            np.savez_compressed(
                out_file,
                threshold=np.array(threshold, dtype=np.float64),
                calibration=cal_score.astype(np.float32),
                iid=iid_score.astype(np.float32),
                holdout=holdout_score.astype(np.float32),
            )
            prediction_index_rows.append({
                "seed":seed,"classifier":kind,"representation":rep,
                "threshold":float(threshold)
            })
            print({"done": key, "threshold": round(float(threshold),6)})

            del model, Xtr, Xcal
            gc.collect()

pd.DataFrame(prediction_index_rows).to_csv(
    OUTPUT_ROOT / "prediction_index.csv", index=False
)
print("Alle 45 Modelle bei 25 % Labels ausgewertet.")

In [ ]:
# ============================================================
# 05 – Hilfsfunktionen für Scores und Fehler
# ============================================================

def load_scores(rep, kind, seed):
    p = PRED_DIR / f"{rep.lower()}_{kind.lower()}_seed{seed}.npz"
    z = np.load(p)
    return {
        "threshold": float(z["threshold"]),
        "iid": np.asarray(z["iid"]),
        "holdout": np.asarray(z["holdout"]),
    }

def scenario_data(rep, kind, seed, scenario):
    z = load_scores(rep, kind, seed)
    source, idx = SCENARIOS[scenario]
    if source == "iid":
        score = z["iid"][idx]
        frame = iid_df.iloc[idx].reset_index(drop=True)
    else:
        score = z["holdout"][idx]
        frame = holdout_df.iloc[idx].reset_index(drop=True)
    y = frame["label"].to_numpy(dtype=int)
    return frame, y, score, z["threshold"]

def safe_jaccard(a, b):
    u = np.logical_or(a,b).sum()
    return float(np.logical_and(a,b).sum() / u) if u else np.nan

In [ ]:
# ============================================================
# 06 – Fehlerkomplementarität BASE ↔ SSL
# ============================================================

rows = []

for seed in SEEDS:
    for kind in CLASSIFIERS:
        for scenario in SCENARIOS:
            _, y, base_score, base_thr = scenario_data(
                "BASE", kind, seed, scenario
            )
            base_pred = base_score >= base_thr
            base_fn = (y==1) & (~base_pred)
            base_fp = (y==0) & base_pred

            for rep in ["DAPT","CONTRASTIVE"]:
                _, _, ssl_score, ssl_thr = scenario_data(
                    rep, kind, seed, scenario
                )
                ssl_pred = ssl_score >= ssl_thr
                ssl_fn = (y==1) & (~ssl_pred)
                ssl_fp = (y==0) & ssl_pred

                rescued_fn = base_fn & ssl_pred
                regressed_tp = (y==1) & base_pred & (~ssl_pred)
                avoided_fp = base_fp & (~ssl_pred)
                new_fp = (y==0) & (~base_pred) & ssl_pred

                rows.append({
                    "seed":seed,
                    "classifier":kind,
                    "scenario":scenario,
                    "ssl_representation":rep,
                    "n_positive":int((y==1).sum()),
                    "n_negative":int((y==0).sum()),
                    "base_fn":int(base_fn.sum()),
                    "ssl_fn":int(ssl_fn.sum()),
                    "common_fn":int((base_fn & ssl_fn).sum()),
                    "rescued_base_fn":int(rescued_fn.sum()),
                    "rescue_rate_of_base_fn":float(
                        rescued_fn.sum()/max(base_fn.sum(),1)
                    ),
                    "regressed_base_tp":int(regressed_tp.sum()),
                    "fn_jaccard":safe_jaccard(base_fn, ssl_fn),
                    "base_fp":int(base_fp.sum()),
                    "ssl_fp":int(ssl_fp.sum()),
                    "avoided_base_fp":int(avoided_fp.sum()),
                    "new_fp_vs_base":int(new_fp.sum()),
                })

comp = pd.DataFrame(rows)
comp.to_csv(OUTPUT_ROOT / "error_complementarity.csv", index=False)

comp_summary = comp.groupby(
    ["classifier","scenario","ssl_representation"], as_index=False
).agg(
    base_fn_mean=("base_fn","mean"),
    rescued_fn_mean=("rescued_base_fn","mean"),
    rescue_rate_mean=("rescue_rate_of_base_fn","mean"),
    rescue_rate_std=("rescue_rate_of_base_fn","std"),
    regressed_tp_mean=("regressed_base_tp","mean"),
    fn_jaccard_mean=("fn_jaccard","mean"),
    new_fp_mean=("new_fp_vs_base","mean"),
)
comp_summary.to_csv(
    OUTPUT_ROOT / "error_complementarity_summary.csv", index=False
)
display(comp_summary)

In [ ]:
# ============================================================
# 07 – Review-Kaskade: BASE Stage 1 + Ranking Stage 2
# ============================================================

cascade_rows = []

for seed in SEEDS:
    for kind in CLASSIFIERS:
        for scenario in SCENARIOS:
            _, y, base_score, base_thr = scenario_data(
                "BASE", kind, seed, scenario
            )
            base_pred = base_score >= base_thr
            stage1_negative = ~base_pred
            base_tp = int(((y==1) & base_pred).sum())
            n_pos = int((y==1).sum())
            base_recall = base_tp / max(n_pos,1)

            ranking_scores = {"BASE_SELF": base_score}
            for rep in ["DAPT","CONTRASTIVE"]:
                ranking_scores[rep] = scenario_data(
                    rep, kind, seed, scenario
                )[2]

            neg_idx = np.flatnonzero(stage1_negative)

            for review_frac in REVIEW_FRACTIONS:
                n_review = max(1, int(math.ceil(review_frac * len(neg_idx))))

                for ranker, score in ranking_scores.items():
                    ordered = neg_idx[np.argsort(score[neg_idx])[::-1]]
                    review_idx = ordered[:n_review]
                    captured = int((y[review_idx]==1).sum())
                    combined_recall = (base_tp + captured) / max(n_pos,1)

                    cascade_rows.append({
                        "seed":seed,
                        "classifier":kind,
                        "scenario":scenario,
                        "ranker":ranker,
                        "review_fraction_of_stage1_negatives":review_frac,
                        "stage1_negative_n":int(len(neg_idx)),
                        "review_n":int(n_review),
                        "review_fraction_of_all_cases":float(n_review/len(y)),
                        "base_tp":base_tp,
                        "base_fn":int(((y==1)&stage1_negative).sum()),
                        "base_recall":base_recall,
                        "captured_base_fn":captured,
                        "capture_rate_of_base_fn":float(
                            captured/max(((y==1)&stage1_negative).sum(),1)
                        ),
                        "review_precision":float(captured/max(n_review,1)),
                        "combined_recall":combined_recall,
                        "recall_gain_pp":float(
                            100*(combined_recall-base_recall)
                        ),
                    })

cascade = pd.DataFrame(cascade_rows)

# Lift gegenüber BASE-Self-Ranking innerhalb exakt derselben Bedingung.
key = [
    "seed","classifier","scenario",
    "review_fraction_of_stage1_negatives"
]
self_ref = cascade[cascade.ranker=="BASE_SELF"][
    key + ["captured_base_fn","combined_recall"]
].rename(columns={
    "captured_base_fn":"self_captured_base_fn",
    "combined_recall":"self_combined_recall",
})
cascade = cascade.merge(self_ref, on=key, how="left")
cascade["extra_captured_vs_self"] = (
    cascade["captured_base_fn"] - cascade["self_captured_base_fn"]
)
cascade["combined_recall_gain_vs_self_pp"] = 100 * (
    cascade["combined_recall"] - cascade["self_combined_recall"]
)

cascade.to_csv(OUTPUT_ROOT / "cascade_results.csv", index=False)

cascade_summary = cascade.groupby(
    [
        "classifier","scenario","ranker",
        "review_fraction_of_stage1_negatives"
    ],
    as_index=False
).agg(
    captured_fn_mean=("captured_base_fn","mean"),
    captured_fn_std=("captured_base_fn","std"),
    capture_rate_mean=("capture_rate_of_base_fn","mean"),
    review_precision_mean=("review_precision","mean"),
    combined_recall_mean=("combined_recall","mean"),
    recall_gain_pp_mean=("recall_gain_pp","mean"),
    extra_captured_vs_self_mean=("extra_captured_vs_self","mean"),
    gain_vs_self_pp_mean=("combined_recall_gain_vs_self_pp","mean"),
)
cascade_summary.to_csv(
    OUTPUT_ROOT / "cascade_summary.csv", index=False
)

display(
    cascade_summary[
        cascade_summary.review_fraction_of_stage1_negatives.eq(0.20)
    ].sort_values(
        ["classifier","scenario","extra_captured_vs_self_mean"],
        ascending=[True,True,False]
    )
)

In [ ]:
# ============================================================
# 08 – 95%-CI über die fünf Seeds für Kaskaden-Lift
# ============================================================

def ci95(series):
    x = np.asarray(series, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return pd.Series({"mean":np.nan,"ci_low":np.nan,"ci_high":np.nan})
    m = x.mean()
    # t(4, .975) ≈ 2.776 für exakt fünf Seeds
    half = 2.776 * x.std(ddof=1) / np.sqrt(len(x))
    return pd.Series({"mean":m,"ci_low":m-half,"ci_high":m+half})

ci_rows = []
for keys, g in cascade[cascade.ranker!="BASE_SELF"].groupby(
    ["classifier","scenario","ranker","review_fraction_of_stage1_negatives"]
):
    ci = ci95(g["combined_recall_gain_vs_self_pp"])
    ci_rows.append({
        "classifier":keys[0],
        "scenario":keys[1],
        "ranker":keys[2],
        "review_fraction":keys[3],
        "gain_vs_self_pp_mean":ci["mean"],
        "gain_vs_self_pp_ci_low":ci["ci_low"],
        "gain_vs_self_pp_ci_high":ci["ci_high"],
    })

cascade_ci = pd.DataFrame(ci_rows)
cascade_ci.to_csv(OUTPUT_ROOT / "cascade_lift_ci95.csv", index=False)
display(cascade_ci)

In [ ]:
# ============================================================
# 09 – Quantitative Fehleranalyse nach Seiteneigenschaften
# ============================================================

def enrich(frame):
    f = frame.copy()
    f["text_chars"] = f["text"].fillna("").astype(str).str.len()
    if "url" in f.columns:
        f["url_chars"] = f["url"].fillna("").astype(str).str.len()
    else:
        f["url_chars"] = np.nan

    if "date" in f.columns:
        dt = pd.to_datetime(f["date"], errors="coerce", utc=True)
        f["year"] = dt.dt.year
    else:
        f["year"] = np.nan

    return f

feature_rows = []

# Fehlerprofile über alle Seeds; drei robuste, einfach interpretierbare Buckets.
for seed in SEEDS:
    for kind in CLASSIFIERS:
        for rep in REPRESENTATIONS:
            for scenario in SCENARIOS:
                frame, y, score, thr = scenario_data(
                    rep, kind, seed, scenario
                )
                frame = enrich(frame)
                pred = score >= thr
                frame["is_fn"] = (y==1) & (~pred)
                frame["is_fp"] = (y==0) & pred
                frame["is_pos"] = y==1
                frame["is_neg"] = y==0

                for feature in ["text_chars","url_chars"]:
                    valid = frame[feature].notna()
                    if valid.sum() < 20:
                        continue
                    try:
                        bins = pd.qcut(
                            frame.loc[valid, feature],
                            q=4, duplicates="drop"
                        )
                    except Exception:
                        continue
                    tmp = frame.loc[valid].copy()
                    tmp["feature_bin"] = bins.astype(str).to_numpy()

                    for b, g in tmp.groupby("feature_bin", observed=True):
                        pos = int(g["is_pos"].sum())
                        neg = int(g["is_neg"].sum())
                        feature_rows.append({
                            "seed":seed,
                            "classifier":kind,
                            "representation":rep,
                            "scenario":scenario,
                            "feature":feature,
                            "feature_bin":str(b),
                            "n":len(g),
                            "fn_rate":float(g["is_fn"].sum()/max(pos,1)),
                            "fp_rate":float(g["is_fp"].sum()/max(neg,1)),
                        })

                if (
                    scenario != "IID"
                    and "min_simhash_distance_to_development" in frame.columns
                ):
                    d = pd.to_numeric(
                        frame["min_simhash_distance_to_development"],
                        errors="coerce"
                    )
                    buckets = pd.cut(
                        d,
                        bins=[-np.inf,2,5,10,np.inf],
                        labels=["<=2","3-5","6-10",">10"],
                    )
                    tmp = frame.copy()
                    tmp["feature_bin"] = buckets
                    for b, g in tmp.dropna(subset=["feature_bin"]).groupby(
                        "feature_bin", observed=True
                    ):
                        pos = int(g["is_pos"].sum())
                        neg = int(g["is_neg"].sum())
                        feature_rows.append({
                            "seed":seed,
                            "classifier":kind,
                            "representation":rep,
                            "scenario":scenario,
                            "feature":"simhash_distance",
                            "feature_bin":str(b),
                            "n":len(g),
                            "fn_rate":float(g["is_fn"].sum()/max(pos,1)),
                            "fp_rate":float(g["is_fp"].sum()/max(neg,1)),
                        })

feature_errors = pd.DataFrame(feature_rows)
feature_errors.to_csv(
    OUTPUT_ROOT / "error_feature_rates.csv", index=False
)

feature_summary = feature_errors.groupby(
    [
        "classifier","representation","scenario",
        "feature","feature_bin"
    ],
    as_index=False
).agg(
    n_mean=("n","mean"),
    fn_rate_mean=("fn_rate","mean"),
    fn_rate_std=("fn_rate","std"),
    fp_rate_mean=("fp_rate","mean"),
    fp_rate_std=("fp_rate","std"),
)
feature_summary.to_csv(
    OUTPUT_ROOT / "error_feature_summary.csv", index=False
)
print({"error_feature_rows": len(feature_errors)})

In [ ]:
# ============================================================
# 10 – Qualitative Fehlerbeispiele (Seed 42)
# ============================================================

sample_rows = []
seed = 42

for kind in CLASSIFIERS:
    for rep in REPRESENTATIONS:
        for scenario in SCENARIOS:
            frame, y, score, thr = scenario_data(rep, kind, seed, scenario)
            frame = frame.reset_index(drop=True).copy()
            pred = score >= thr

            # harte False Negatives: niedrigster Phishing-Score
            fn_idx = np.flatnonzero((y==1) & (~pred))
            fn_idx = fn_idx[np.argsort(score[fn_idx])][:10] if len(fn_idx) else []

            # harte False Positives: höchster Benign-Score
            fp_idx = np.flatnonzero((y==0) & pred)
            fp_idx = fp_idx[np.argsort(score[fp_idx])[::-1]][:10] if len(fp_idx) else []

            for error_type, indices in [("FN",fn_idx),("FP",fp_idx)]:
                for i in indices:
                    r = frame.iloc[int(i)]
                    sample_rows.append({
                        "seed":seed,
                        "classifier":kind,
                        "representation":rep,
                        "scenario":scenario,
                        "error_type":error_type,
                        "score":float(score[i]),
                        "threshold":float(thr),
                        "sha256":str(r.get("sha256","")),
                        "domain":str(r.get("domain","")),
                        "url":str(r.get("url",""))[:500],
                        "date":str(r.get("date","")),
                        "template_hash":str(r.get("template_hash","")),
                        "near_duplicate_to_development":r.get(
                            "near_duplicate_to_development", np.nan
                        ),
                        "min_simhash_distance_to_development":r.get(
                            "min_simhash_distance_to_development", np.nan
                        ),
                        "text_preview":str(r.get("text",""))[:350].replace("\n"," "),
                    })

samples = pd.DataFrame(sample_rows)
samples.to_csv(OUTPUT_ROOT / "qualitative_error_samples_seed42.csv", index=False)
print({"qualitative_error_samples": len(samples)})

In [ ]:
# ============================================================
# 11 – Abschluss und ZIP
# ============================================================

marker = {
    "status":"COMPLETE",
    "analysis":"25pct cascade + error complementarity",
    "stage1":"BASE at calibration-derived 0.5% target FPR",
    "stage2_rankers":["BASE_SELF","DAPT","CONTRASTIVE"],
    "review_fractions":REVIEW_FRACTIONS,
    "classifiers":CLASSIFIERS,
    "seeds":SEEDS,
    "scenarios":list(SCENARIOS.keys()),
}
(OUTPUT_ROOT / "CASCADE_ERROR_ANALYSIS_COMPLETE.json").write_text(
    json.dumps(marker, indent=2), encoding="utf-8"
)

archive = shutil.make_archive(
    "/kaggle/working/phreshphish_cascade_error_analysis_25pct",
    "zip",
    root_dir=OUTPUT_ROOT,
)

print(marker)
print({"zip": archive})

## Zentrale Ergebnisdateien

- `cascade_results.csv`: vollständige Kaskadenergebnisse je Seed
- `cascade_summary.csv`: Mittelwerte über fünf Seeds
- `cascade_lift_ci95.csv`: gepaarter Lift gegenüber BASE-Self-Ranking
- `error_complementarity.csv`: Fehlerüberlappung BASE ↔ DAPT/Contrastive
- `error_complementarity_summary.csv`: aggregierte Rescue-/Regression-Raten
- `error_feature_rates.csv`: Fehlerquoten nach Seitenmerkmalen
- `error_feature_summary.csv`: aggregierte Fehlerprofile
- `qualitative_error_samples_seed42.csv`: konkrete harte FN-/FP-Beispiele
- `prediction_index.csv`: verwendete Schwellenwerte
- `scores/`: reproduzierbare Rohscores der 45 Modelle
- `CASCADE_ERROR_ANALYSIS_COMPLETE.json`: Completion-Marker

Die Kaskade ist bewusst eine **nachgelagerte Triageanalyse** und verändert den FINAL FREEZE nicht.